In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

import os, sys
os.chdir('..')

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.samplers.StickyAutomaticBoomerangSampler import StickyAutomaticBoomerangSampler
from sazz.models.glm import (
    make_linear_regression,
    make_logistic_regression,
    make_glm,
    make_kappa_vector_glm,
    predict_linear,
)

In [ ]:
import numpy as np
import torch

rng = np.random.default_rng(0)

# ───────────────────────────────────────────────────────────────
# 1. Linear regression data
#    y = X @ beta_true + noise,  mostly sparse coefficients
# ───────────────────────────────────────────────────────────────
N_lin, D_lin = 200, 10
X_lin = rng.normal(size=(N_lin, D_lin))
beta_true_lin = np.zeros(D_lin)
beta_true_lin[[0, 3, 7]] = [1.5, -2.0, 0.8]  # only 3 true signals
intercept_lin = 0.5
y_lin = X_lin @ beta_true_lin + intercept_lin + 0.3 * rng.normal(size=N_lin)

# Standardise features (don't standardise y for linear — keep interpretable)
X_lin = (X_lin - X_lin.mean(0)) / X_lin.std(0)

X_lin_t = torch.tensor(X_lin, dtype=torch.float64)
y_lin_t = torch.tensor(y_lin, dtype=torch.float64)

print(f"Linear: X {X_lin.shape}, y {y_lin.shape}, true nonzero: {(beta_true_lin != 0).sum()}/{D_lin}")


# ───────────────────────────────────────────────────────────────
# 2. Logistic regression data (binary)
# ───────────────────────────────────────────────────────────────
N_log, D_log = 300, 8
X_log = rng.normal(size=(N_log, D_log))
beta_true_log = np.zeros(D_log)
beta_true_log[[1, 4]] = [2.0, -1.5]
intercept_log = -0.3
logits_log = X_log @ beta_true_log + intercept_log
probs_log = 1.0 / (1.0 + np.exp(-logits_log))
y_log = rng.binomial(1, probs_log)

X_log = (X_log - X_log.mean(0)) / X_log.std(0)
X_log_t = torch.tensor(X_log, dtype=torch.float64)
y_log_t = torch.tensor(y_log, dtype=torch.long)

print(f"Logistic (binary): X {X_log.shape}, class balance: {y_log.mean():.2f}")


# ───────────────────────────────────────────────────────────────
# 3. Logistic regression data (multinomial, 3 classes)
# ───────────────────────────────────────────────────────────────
N_multi, D_multi, K_multi = 300, 6, 3
X_multi = rng.normal(size=(N_multi, D_multi))
# Random coefficients per class
beta_true_multi = rng.normal(scale=1.5, size=(K_multi, D_multi + 1))
X_multi_aug = np.hstack([np.ones((N_multi, 1)), X_multi])
logits_multi = X_multi_aug @ beta_true_multi.T              # [N, K]
probs_multi = np.exp(logits_multi - logits_multi.max(1, keepdims=True))
probs_multi /= probs_multi.sum(1, keepdims=True)
y_multi = np.array([rng.choice(K_multi, p=p) for p in probs_multi])

X_multi = (X_multi - X_multi.mean(0)) / X_multi.std(0)
X_multi_t = torch.tensor(X_multi, dtype=torch.float64)
y_multi_t = torch.tensor(y_multi, dtype=torch.long)

print(f"Logistic (multinomial): X {X_multi.shape}, "
      f"class counts: {np.bincount(y_multi)}")


# ───────────────────────────────────────────────────────────────
# 4. Poisson regression (count data, log link)
# ───────────────────────────────────────────────────────────────
N_pois, D_pois = 200, 6
X_pois = rng.normal(size=(N_pois, D_pois))
beta_true_pois = np.zeros(D_pois)
beta_true_pois[[0, 2]] = [0.4, -0.3]   # keep modest to avoid huge rates
intercept_pois = 1.0
log_rate = X_pois @ beta_true_pois + intercept_pois
y_pois = rng.poisson(np.exp(log_rate))

X_pois = (X_pois - X_pois.mean(0)) / X_pois.std(0)
X_pois_t = torch.tensor(X_pois, dtype=torch.float64)
y_pois_t = torch.tensor(y_pois, dtype=torch.float64)

print(f"Poisson: X {X_pois.shape}, y range: [{y_pois.min()}, {y_pois.max()}], "
      f"mean={y_pois.mean():.2f}")


# ───────────────────────────────────────────────────────────────
# 5. Gamma regression (positive continuous data, log link)
# ───────────────────────────────────────────────────────────────
N_gam, D_gam = 200, 5
X_gam = rng.normal(size=(N_gam, D_gam))
beta_true_gam = np.array([0.3, -0.2, 0.0, 0.5, 0.0])
intercept_gam = 0.5
log_mu = X_gam @ beta_true_gam + intercept_gam
mu_gam = np.exp(log_mu)
shape_gam = 2.0
y_gam = rng.gamma(shape=shape_gam, scale=mu_gam / shape_gam)  # mean = mu

X_gam = (X_gam - X_gam.mean(0)) / X_gam.std(0)
X_gam_t = torch.tensor(X_gam, dtype=torch.float64)
y_gam_t = torch.tensor(y_gam, dtype=torch.float64)

print(f"Gamma: X {X_gam.shape}, y range: [{y_gam.min():.2f}, {y_gam.max():.2f}], "
      f"mean={y_gam.mean():.2f}")

In [ ]:
# Linear regression
target_linreg = make_linear_regression(
    X_lin_t, y_lin_t,
    prior_std=1.0,
    intercept_prior_std=10.0,
    noise_std=0.5,
)

sampler_linreg = AutomaticBoomerangSampler(
    grad_target=target_linreg.grad_target, D=target_linreg.D, refresh_rate=1.0,
    thinning="pli",
)
sampler_linreg.preprocess(x_ref=target_linreg.x_ref, Sigma_inv=target_linreg.Sigma_inv)


# Sticky version
kappa_automatic = make_kappa_vector_glm(target_linreg.D, kappa_coef=0.1)
kappa_manual = torch.tensor([1.0000e+06, 1.0000e-00, 1.0000e-01, 1.0000e-01, 1.0000e-00, 1.0000e-01,
        1.0000e-01, 1.0000e-01, 1.0000e-00, 1.0000e-01, 1.0000e-01],
       dtype=torch.float64)
sampler_linreg_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target_linreg.grad_target, D=target_linreg.D, refresh_rate=1.0,
    kappa=kappa_manual, thinning="pli",
)
sampler_linreg_sticky.preprocess(x_ref=target_linreg.x_ref, Sigma_inv=target_linreg.Sigma_inv)

In [ ]:
# --- Sample ---
N_SKELETON = 100_000
result_linreg = sampler_linreg.sample(N=N_SKELETON, diagnostics=True)

result_linreg_sticky = sampler_linreg_sticky.sample(N=N_SKELETON, diagnostics=True)

In [ ]:
from sazz.utils.sampling import resample_pdmp_path, resample_pdmp_path_sticky

N_RESAMPLE = 50_000
BURNIN_FRAC = 0.1

# --- Resample both chains ---
samples_linreg = resample_pdmp_path(
    result_linreg["positions"].cpu().numpy(),
    result_linreg["velocities"].cpu().numpy(),
    result_linreg["times"].cpu().numpy(),
    target_linreg.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC,
)
samples_linreg_sticky = resample_pdmp_path_sticky(
    result_linreg_sticky["positions"].cpu().numpy(),
    result_linreg_sticky["velocities"].cpu().numpy(),
    result_linreg_sticky["times"].cpu().numpy(),
    target_linreg.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC,
)

# --- Per-coefficient summary ---
# True coefficients: intercept + beta_true_lin
true_coefs = np.concatenate([[intercept_lin], beta_true_lin])

def summarise(samples, name):
    means = samples.mean(0)
    stds  = samples.std(0)
    sparsity = np.mean(np.abs(samples) < 1e-8, axis=0)
    print(f"\n=== {name} ===")
    print(f"{'coord':<6} {'true':>8} {'mean':>8} {'std':>8} {'P(0)':>6}")
    print("-" * 42)
    labels = ["β_0 (int)"] + [f"β_{i+1}" for i in range(len(means) - 1)]
    for i, (lbl, tc, m, s, p) in enumerate(zip(labels, true_coefs, means, stds, sparsity)):
        marker = " *" if tc != 0 else ""
        print(f"{lbl:<9} {tc:>8.3f} {m:>8.3f} {s:>8.3f} {p:>6.2f}{marker}")

summarise(samples_linreg, "Boomerang")
summarise(samples_linreg_sticky, "Sticky Boomerang")

# --- Quick recovery metric: correlation between true and estimated ---
print("\n=== Recovery ===")
for name, samp in [("Boomerang", samples_linreg), ("Sticky", samples_linreg_sticky)]:
    est = samp.mean(0)
    corr = np.corrcoef(true_coefs, est)[0, 1]
    rmse = np.sqrt(np.mean((true_coefs - est) ** 2))
    print(f"{name:20s}: corr={corr:.3f}, RMSE(coefs)={rmse:.3f}")

## LOGREG

In [ ]:
beta_true_log

In [ ]:
# Linear regression
target_logreg = make_logistic_regression(
    X_log_t, y_log_t,
    prior_std=1.0,
    intercept_prior_std=10.0,
)

sampler_logreg = AutomaticBoomerangSampler(
    grad_target=target_logreg.grad_target, D=target_logreg.D, refresh_rate=1.0,
    thinning="pli",
)
sampler_logreg.preprocess(x_ref=target_logreg.x_ref, Sigma_inv=target_logreg.Sigma_inv)


# Sticky version
kappa_logreg_automatic = make_kappa_vector_glm(target_logreg.D, kappa_coef=0.1)
kappa_logreg_manual = torch.tensor([1.0000e+06, 1.0000e-00, 1.0000e-01, 1.0000e-01, 1.0000e-00, 1.0000e-01,
        1.0000e-01, 1.0000e-01, 1.0000e-01],
       dtype=torch.float64)
sampler_logreg_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target_logreg.grad_target, D=target_logreg.D, refresh_rate=1.0,
    kappa=kappa_logreg_manual, thinning="pli",
)
sampler_logreg_sticky.preprocess(x_ref=target_logreg.x_ref, Sigma_inv=target_logreg.Sigma_inv)

In [ ]:
# --- Sample ---
N_SKELETON = 100_000
result_logreg = sampler_logreg.sample(N=N_SKELETON, diagnostics=True)

result_logreg_sticky = sampler_logreg_sticky.sample(N=N_SKELETON, diagnostics=True)

In [ ]:
from sazz.utils.sampling import resample_pdmp_path, resample_pdmp_path_sticky

N_RESAMPLE = 50_000
BURNIN_FRAC = 0.1

# --- Resample both chains ---
samples_logreg = resample_pdmp_path(
    result_logreg["positions"].cpu().numpy(),
    result_logreg["velocities"].cpu().numpy(),
    result_logreg["times"].cpu().numpy(),
    target_logreg.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC,
)
samples_logreg_sticky = resample_pdmp_path_sticky(
    result_logreg_sticky["positions"].cpu().numpy(),
    result_logreg_sticky["velocities"].cpu().numpy(),
    result_logreg_sticky["times"].cpu().numpy(),
    target_logreg.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC,
)

# --- Per-coefficient summary ---
# True coefficients: intercept + beta_true_lin
true_coefs = np.concatenate([[intercept_log], beta_true_log])

def summarise(samples, name):
    means = samples.mean(0)
    stds  = samples.std(0)
    sparsity = np.mean(np.abs(samples) < 1e-8, axis=0)
    print(f"\n=== {name} ===")
    print(f"{'coord':<6} {'true':>8} {'mean':>8} {'std':>8} {'P(0)':>6}")
    print("-" * 42)
    labels = ["β_0 (int)"] + [f"β_{i+1}" for i in range(len(means) - 1)]
    for i, (lbl, tc, m, s, p) in enumerate(zip(labels, true_coefs, means, stds, sparsity)):
        marker = " *" if tc != 0 else ""
        print(f"{lbl:<9} {tc:>8.3f} {m:>8.3f} {s:>8.3f} {p:>6.2f}{marker}")

summarise(samples_logreg, "Boomerang")
summarise(samples_logreg_sticky, "Sticky Boomerang")

# --- Quick recovery metric: correlation between true and estimated ---
print("\n=== Recovery ===")
for name, samp in [("Boomerang", samples_logreg), ("Sticky", samples_logreg_sticky)]:
    est = samp.mean(0)
    corr = np.corrcoef(true_coefs, est)[0, 1]
    rmse = np.sqrt(np.mean((true_coefs - est) ** 2))
    print(f"{name:20s}: corr={corr:.3f}, RMSE(coefs)={rmse:.3f}")